# Qwen3-4B-Instruct + LoRA 파인튜닝 실습

목표: 원본 Qwen 모델이 모르는 `PERSOAI` 내부 교육 코드를 LoRA로 학습시킨 뒤, 원본 모델과 파인튜닝 모델의 답변 차이를 비교합니다.

런타임: Google Colab GPU 권장

In [ ]:
!pip install -q -U transformers datasets accelerate peft trl

In [ ]:
# ============================================================
# Qwen3-4B-Instruct + LoRA Fine-tuning Example
# Purpose: Teach a model private PERSOAI course-code meanings
# Recommended environment: Google Colab with GPU

In [ ]:
# ============================================================
# 1. Install packages

In [ ]:
# ============================================================
# Run this cell in Colab first:
# !pip install -q -U transformers datasets accelerate peft trl

import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

In [ ]:
# ============================================================
# 2. Model settings

In [ ]:
# ============================================================
model_id = "Qwen/Qwen3-4B-Instruct-2507"
adapter_path = "./qwen3-persoai-code-lora"


def load_base_model():
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )

    model.config.use_cache = False
    model.eval()
    return tokenizer, model

In [ ]:
# ============================================================
# 3. Common inference function

In [ ]:
# ============================================================
def ask_model(model, tokenizer, question):
    messages = [
        {
            "role": "system",
            "content": "너는 PERSOAI 교육 과정의 내부 코드 의미를 정확히 설명하는 AI 조교다. 모르는 것은 추측하지 말고 모른다고 답한다.",
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=False,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# ============================================================
# 4. Fine-tuning dataset

In [ ]:
# ============================================================
train_data = [
    {
        "instruction": "PERSOAI-STT-01 코드를 설명해줘.",
        "output": "PERSOAI-STT-01은 음성 인식 기초 실습을 의미합니다. 학생은 마이크로 음성을 입력하고, Whisper 모델을 사용해 한국어 문장으로 변환하는 기본 과정을 학습합니다.",
    },
    {
        "instruction": "PERSOAI-STT-02 코드를 설명해줘.",
        "output": "PERSOAI-STT-02는 실시간 음성 인식 실습을 의미합니다. 학생은 마이크 입력을 받아 짧은 단위로 음성을 처리하고, 실시간으로 텍스트를 출력하는 방법을 학습합니다.",
    },
    {
        "instruction": "PERSOAI-TTS-01 코드를 설명해줘.",
        "output": "PERSOAI-TTS-01은 한국어 음성 합성 기초 실습을 의미합니다. 학생은 입력한 문장을 TTS 모델을 이용해 한국어 음성으로 변환하는 과정을 학습합니다.",
    },
    {
        "instruction": "PERSOAI-TTS-02 코드를 설명해줘.",
        "output": "PERSOAI-TTS-02는 사용자 음성 스타일을 반영한 음성 합성 실습을 의미합니다. 학생은 샘플 음성을 이용해 비슷한 말투의 음성을 생성하는 방법을 학습합니다.",
    },
    {
        "instruction": "PERSOAI-RAG-01 코드를 설명해줘.",
        "output": "PERSOAI-RAG-01은 문서 기반 질의응답 기초 실습을 의미합니다. 학생은 PDF나 텍스트 문서를 벡터DB에 저장하고, 질문과 관련된 내용을 검색해 답변하는 구조를 학습합니다.",
    },
    {
        "instruction": "PERSOAI-RAG-02 코드를 설명해줘.",
        "output": "PERSOAI-RAG-02는 검색 증강 생성 실습을 의미합니다. 학생은 사용자의 질문에 대해 관련 문서를 먼저 찾고, 찾은 내용을 바탕으로 LLM이 답변하도록 구성합니다.",
    },
    {
        "instruction": "PERSOAI-LLM-01 코드를 설명해줘.",
        "output": "PERSOAI-LLM-01은 로컬 LLM 실행 기초 실습을 의미합니다. 학생은 Qwen, Llama 같은 사전학습 모델을 불러오고 간단한 질문에 답변을 생성하는 방법을 학습합니다.",
    },
    {
        "instruction": "PERSOAI-LLM-02 코드를 설명해줘.",
        "output": "PERSOAI-LLM-02는 LoRA 파인튜닝 기초 실습을 의미합니다. 학생은 작은 데이터셋을 이용해 모델의 답변 스타일이나 특정 코드 해석 능력을 추가 학습시킵니다.",
    },
    {
        "instruction": "PERSOAI-CHAT-01 코드를 설명해줘.",
        "output": "PERSOAI-CHAT-01은 기본 챗봇 구현 실습을 의미합니다. 학생은 사용자의 질문을 입력받고 LLM이 답변을 생성하는 간단한 대화형 프로그램을 만듭니다.",
    },
    {
        "instruction": "PERSOAI-CHAT-02 코드를 설명해줘.",
        "output": "PERSOAI-CHAT-02는 음성 챗봇 구현 실습을 의미합니다. 학생은 STT, LLM, TTS를 연결해 사용자가 말하면 AI가 듣고 답하고 다시 음성으로 말하는 구조를 학습합니다.",
    },
]


def build_dataset(tokenizer):
    def make_chat_text(example):
        messages = [
            {
                "role": "system",
                "content": "너는 PERSOAI 교육 과정의 내부 코드 의미를 정확히 설명하는 AI 조교다.",
            },
            {
                "role": "user",
                "content": example["instruction"],
            },
            {
                "role": "assistant",
                "content": example["output"],
            },
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        return {"text": text}

    dataset = Dataset.from_list(train_data)
    dataset = dataset.map(make_chat_text)
    return dataset

In [ ]:
# ============================================================
# 5. LoRA config

In [ ]:
# ============================================================
def get_lora_config():
    return LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )

In [ ]:
# ============================================================
# 6. Fine-tuning

In [ ]:
# ============================================================
def train_lora():
    tokenizer, model = load_base_model()
    dataset = build_dataset(tokenizer)

    training_args = SFTConfig(
        output_dir=adapter_path,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        num_train_epochs=10,
        logging_steps=1,
        save_strategy="epoch",
        max_seq_length=512,
        bf16=True,
        fp16=False,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        peft_config=get_lora_config(),
        processing_class=tokenizer,
    )

    trainer.train()
    trainer.model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)

    print(f"LoRA adapter saved to: {adapter_path}")

In [ ]:
# ============================================================
# 7. Load fine-tuned model

In [ ]:
# ============================================================
def load_fine_tuned_model():
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )

    fine_tuned_model = PeftModel.from_pretrained(base_model, adapter_path)
    fine_tuned_model.eval()
    return tokenizer, fine_tuned_model

In [ ]:
# ============================================================
# 8. Comparison test

In [ ]:
# ============================================================
def compare_models():
    tokenizer, base_model = load_base_model()
    _, fine_tuned_model = load_fine_tuned_model()

    test_questions = [
        "PERSOAI-STT-01 코드를 설명해줘.",
        "PERSOAI-TTS-02 코드를 설명해줘.",
        "PERSOAI-RAG-01 코드를 설명해줘.",
        "PERSOAI-LLM-02 코드를 설명해줘.",
        "PERSOAI-CHAT-02 코드를 설명해줘.",
    ]

    for q in test_questions:
        print("=" * 80)
        print("질문:", q)

        print("\n[원본 모델 답변]")
        print(ask_model(base_model, tokenizer, q))

        print("\n[파인튜닝 모델 답변]")
        print(ask_model(fine_tuned_model, tokenizer, q))

In [ ]:
# ============================================================
# 9. Main execution

## 1) 원본 모델 먼저 테스트

In [ ]:
tokenizer, base_model = load_base_model()
print(ask_model(base_model, tokenizer, "PERSOAI-STT-01 코드를 설명해줘."))

## 2) LoRA 파인튜닝 실행

In [ ]:
train_lora()

## 3) 원본 모델과 파인튜닝 모델 비교

In [ ]:
compare_models()